# Combined Interactive Web Map — Gentrification Copenhagen

Produces a single self-contained HTML file (`results/figures/combined_web_map.html`) with:
- OpenStreetMap basemap
- Grouped checkbox layer panel (top-right)
- Collapsible legend (bottom-right)

**Layers**
| Group | Layers |
|---|---|
| Model Predictions | XGBoost 2020, LSTM 2020 |
| Soft Ensemble | 2020, 2025, 2030, 2035 |
| Movement of Gentrification | Gentrification Timeline (when first predicted) |
| Ground Truth | Combined Ground Truth (Low / Medium / High) |
| Boroughs | Quarters / City districts |

**Data sources**
- `data/interim/qgis_to_html/difference_models.gpkg` — neighbourhood cluster geometry + all prediction columns
- `data/processed/classified_gt/classified_combined.gpkg` — combined ground-truth classification
- `data/interim/qgis_to_html/quaters.gpkg` — Copenhagen borough boundaries

In [34]:
import os
import json
import warnings

import geopandas as gpd
import folium

warnings.filterwarnings('ignore')

# ── Paths (relative from notebooks/MAPS/) ──────────────────────────────────────
ROOT     = os.path.join('..', '..')
GPKG_DIR = os.path.join(ROOT, 'data', 'interim', 'qgis_to_html')
GT_DIR   = os.path.join(ROOT, 'data', 'processed', 'classified_gt')
OUT_DIR  = os.path.join(ROOT)
os.makedirs(OUT_DIR, exist_ok=True)

print('folium version:', folium.__version__)
print('Output directory:', os.path.abspath(OUT_DIR))

folium version: 0.20.0
Output directory: c:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model


In [35]:
# ── Load & reproject data ───────────────────────────────────────────────────────

# Neighbourhood cluster polygons + all model predictions (EPSG 25832 → 4326)
gdf_models = (
    gpd.read_file(os.path.join(GPKG_DIR, 'difference_models.gpkg'), layer='difference_models')
    .to_crs(epsg=4326)
)

# Combined ground-truth classification (EPSG 25832 → 4326)
gdf_gt = (
    gpd.read_file(os.path.join(GT_DIR, 'classified_combined.gpkg'), layer='classified_combined')
    .to_crs(epsg=4326)
)

# Borough / quarter boundaries (already WGS84, CRS tag missing in file)
gdf_q = gpd.read_file(os.path.join(GPKG_DIR, 'quaters.gpkg'), layer='quater')
if gdf_q.crs is None:
    gdf_q = gdf_q.set_crs(epsg=4326)

print(f'Neighbourhood clusters : {len(gdf_models):>5} polygons')
print(f'Ground truth           : {len(gdf_gt):>5} polygons')
print(f'Borough quarters       : {len(gdf_q):>5} polygons')
print()

# Prediction columns are stored as string objects ('0'/'1'/None) — normalise to int
PRED_COLS = [
    'ensemble_predictions_xgb_pred',
    'ensemble_predictions_lstm_pred',
    'ensemble_predictions_soft_pred',
    'ensemble_future_2025_prediction',
    'ensemble_future_2030_prediction',
    'ensemble_future_2035_prediction',
]
for col in PRED_COLS:
    gdf_models[col] = gdf_models[col].apply(
        lambda v: int(v) if v is not None and str(v).strip() not in ('', 'None', 'nan') else None
    )

# ── Derive "first year predicted as gentrified" column ─────────────────────────
def _first_gentrified(row):
    """Return the earliest year the soft-ensemble predicts gentrification, or 'Never'."""
    if row['ensemble_predictions_soft_pred'] == 1:
        return '2020'
    if row['ensemble_future_2025_prediction'] == 1:
        return '2025'
    if row['ensemble_future_2030_prediction'] == 1:
        return '2030'
    if row['ensemble_future_2035_prediction'] == 1:
        return '2035'
    return 'Never'

gdf_models['first_gentrified_year'] = gdf_models.apply(_first_gentrified, axis=1)
print('Movement layer — first_gentrified_year distribution:')
print(gdf_models['first_gentrified_year'].value_counts().to_string())
print()

# ── Merge ground truth into models GeoDataFrame ────────────────────────────────
gdf_merged = gdf_models.merge(
    gdf_gt[['cluster_id', 'combined_class', 'avg_mean']],
    on='cluster_id',
    how='left',
)
matched = gdf_merged['combined_class'].notna().sum()
print(f'Ground truth matched: {matched}/{len(gdf_merged)} polygons ({matched/len(gdf_merged)*100:.1f} %)')
print('combined_class distribution:')
print(gdf_merged['combined_class'].value_counts().to_string())

Neighbourhood clusters :  2232 polygons
Ground truth           :  2232 polygons
Borough quarters       :    18 polygons

Movement layer — first_gentrified_year distribution:
first_gentrified_year
2020     1016
Never     929
2035      107
2030       91
2025       89

Ground truth matched: 2232/2232 polygons (100.0 %)
combined_class distribution:
combined_class
Low       871
Medium    744
High      617


In [36]:
# ── Encode GeoJSON (single shared geometry blob for all neighbourhood layers) ───
KEEP_COLS = [
    'cluster_id',
    'ensemble_predictions_xgb_pred',
    'ensemble_predictions_lstm_pred',
    'ensemble_predictions_soft_pred',
    'ensemble_future_2025_prediction',
    'ensemble_future_2030_prediction',
    'ensemble_future_2035_prediction',
    'first_gentrified_year',
    'combined_class',
    'avg_mean',
    'geometry',
]
geojson_nbhd     = json.loads(gdf_merged[KEEP_COLS].to_json())
geojson_quarters = json.loads(gdf_q[['navn', 'geometry']].to_json())

print(f'Neighbourhood GeoJSON features : {len(geojson_nbhd["features"])}')
print(f'Quarter GeoJSON features       : {len(geojson_quarters["features"])}')

# ── Build Folium base map ───────────────────────────────────────────────────────
m = folium.Map(
    location=[55.685, 12.570],
    zoom_start=11,
    tiles='OpenStreetMap',
    control_scale=True,
)

# ── CSS ─────────────────────────────────────────────────────────────────────────
panel_css = """
<style>
/* ── Layer control panel (top-right) ──────────────────── */
#glc-panel {
    position: fixed; top: 80px; right: 10px; z-index: 9999;
    background: rgba(255,255,255,0.97);
    padding: 10px 14px 12px; border-radius: 8px;
    box-shadow: 0 2px 12px rgba(0,0,0,.28);
    font-size: 13px; font-family: Arial, sans-serif;
    min-width: 220px; max-height: 82vh; overflow-y: auto;
}
#glc-panel .ph {
    display: flex; justify-content: space-between;
    align-items: center; margin-bottom: 5px;
}
#glc-panel .gl {
    font-weight: bold; margin: 8px 0 3px;
    padding-bottom: 2px; border-bottom: 1px solid #ddd; color: #333;
}
#glc-panel label {
    display: flex; align-items: center; gap: 6px;
    margin: 3px 0; cursor: pointer;
}
#glc-panel input[type=checkbox] { cursor: pointer; }
#glc-panel .tb {
    background: none; border: none; cursor: pointer;
    font-size: 14px; color: #555; padding: 0 2px;
}
/* ── Opacity slider rows ───────────────────────────────── */
.op-row {
    display: none;
    align-items: center; gap: 5px;
    margin: 0 0 4px 22px;
    font-size: 11px; color: #666;
}
.op-row input[type=range] {
    flex: 1; cursor: pointer; height: 4px;
    accent-color: #666;
}
.op-val { min-width: 30px; text-align: right; color: #444; }
/* ── Legend (bottom-left) ──────────────────────────────── */
#glc-legend {
    position: fixed; bottom: 30px; left: 10px; z-index: 9999;
    background: rgba(255,255,255,0.97);
    padding: 10px 14px; border-radius: 8px;
    box-shadow: 0 2px 12px rgba(0,0,0,.28);
    font-size: 12px; font-family: Arial, sans-serif;
    max-width: 200px; line-height: 1.7;
}
#glc-legend .lh {
    font-weight: bold; font-size: 13px; cursor: pointer; margin-bottom: 3px;
}
#glc-legend .lgh { font-weight: bold; margin: 6px 0 2px; }
.sw {
    display: inline-block; width: 13px; height: 13px;
    border-radius: 2px; margin-right: 4px;
    vertical-align: middle; border: 1px solid rgba(0,0,0,.15);
}
</style>
"""

# ── Helper: build one layer-item block (checkbox + opacity slider) ──────────────
def _layer_item(lid, label, default_op=70):
    return (
        f'<label><input type="checkbox" id="cb_{lid}" '
        f'onchange="toggleLayer(\'{lid}\',this.checked)"> {label}</label>\n'
        f'    <div class="op-row" id="op-row-{lid}">\n'
        f'      <span>Opacity:</span>\n'
        f'      <input type="range" min="10" max="100" value="{default_op}"\n'
        f'        oninput="setOpacity(\'{lid}\',this.value/100);'
        f'document.getElementById(\'op-val-{lid}\').textContent=this.value+\'%\'">\n'
        f'      <span class="op-val" id="op-val-{lid}">{default_op}%</span>\n'
        f'    </div>'
    )

# ── HTML: layer panel ───────────────────────────────────────────────────────────
panel_html = f"""
<div id="glc-panel">
  <div class="ph">
    <b>&#x1F5FA;&nbsp;Layers</b>
    <button class="tb" onclick="
      var b=document.getElementById('glc-body');
      b.style.display=b.style.display==='none'?'block':'none';
      this.textContent=b.style.display==='none'?'&#9660;':'&#9650;';
    ">&#9650;</button>
  </div>
  <div id="glc-body">

    <div class="gl">Model Predictions</div>
    {_layer_item('xgb_2020',  'XGBoost 2020')}
    {_layer_item('lstm_2020', 'LSTM 2020')}

    <div class="gl">Soft Ensemble</div>
    {_layer_item('soft_2020', '2020')}
    {_layer_item('soft_2025', '2025')}
    {_layer_item('soft_2030', '2030')}
    {_layer_item('soft_2035', '2035')}

    <div class="gl">Movement of Gentrification</div>
    {_layer_item('movement',  'Gentrification Timeline')}

    <div class="gl">Ground Truth</div>
    {_layer_item('gt',        'Combined Ground Truth')}

    <div class="gl">Boroughs</div>
    {_layer_item('quarters',  'Quarters', default_op=100)}

  </div>
</div>
"""

# ── HTML: legend (body populated dynamically by JS) ─────────────────────────────
legend_html = """
<div id="glc-legend">
  <div class="lh" onclick="
    var b=document.getElementById('leg-body');
    b.style.display=b.style.display==='none'?'block':'none';
    this.firstChild.nodeValue=b.style.display==='none'?'▶ Legend':'▼ Legend';
  ">▼ Legend</div>
  <div id="leg-body"><i style="color:#888;font-size:11px">No layers selected</i></div>
</div>
"""

# ── JavaScript: layer engine ────────────────────────────────────────────────────
js_code = f"""
<script>
(function () {{
    function init() {{
        var leafletMap = null;
        for (var k in window) {{
            if (k.indexOf('map_') === 0) {{
                try {{ if (window[k] instanceof L.Map) {{ leafletMap = window[k]; break; }} }}
                catch (e) {{}}
            }}
        }}
        if (!leafletMap) {{ setTimeout(init, 150); return; }}

        /* ── Embedded GeoJSON ───────────────────────────────────── */
        var NBHD_DATA = {json.dumps(geojson_nbhd)};
        var QRTR_DATA = {json.dumps(geojson_quarters)};

        /* ── Color maps (from QGIS QML files) ──────────────────── */
        var BINARY   = {{ 1: '#E15F1D', 0: '#0C8A00' }};
        var MOVEMENT = {{ '2020':'#E15F1D','2025':'#ED7B40','2030':'#FBA071','2035':'#FFCFB6','Never':'#0C8A00' }};
        var GTCOL    = {{ 'Low':'#0C8A00','Medium':'#F0E442','High':'#E15F1D' }};

        /* ── Layer configuration ────────────────────────────────── */
        var CONFIGS = [
            {{ id:'xgb_2020',  col:'ensemble_predictions_xgb_pred',   type:'binary',       label:'XGBoost Prediction'    }},
            {{ id:'lstm_2020', col:'ensemble_predictions_lstm_pred',  type:'binary',       label:'LSTM Prediction'       }},
            {{ id:'soft_2020', col:'ensemble_predictions_soft_pred',  type:'binary',       label:'Soft Ensemble 2020'    }},
            {{ id:'soft_2025', col:'ensemble_future_2025_prediction', type:'binary',       label:'Soft Ensemble 2025'    }},
            {{ id:'soft_2030', col:'ensemble_future_2030_prediction', type:'binary',       label:'Soft Ensemble 2030'    }},
            {{ id:'soft_2035', col:'ensemble_future_2035_prediction', type:'binary',       label:'Soft Ensemble 2035'    }},
            {{ id:'movement',  col:'first_gentrified_year',           type:'movement',     label:'First Gentrified Year' }},
            {{ id:'gt',        col:'combined_class',                  type:'ground_truth', label:'Ground Truth Class'    }},
        ];
        var CFG = {{}};
        CONFIGS.forEach(function(c) {{ CFG[c.id] = c; }});

        /* ── State ──────────────────────────────────────────────── */
        var ACTIVE  = {{}};
        var OPACITY = {{
            xgb_2020:0.70, lstm_2020:0.70, soft_2020:0.70, soft_2025:0.70,
            soft_2030:0.70, soft_2035:0.70, movement:0.70, gt:0.70, quarters:1.00
        }};

        /* ── Helpers ────────────────────────────────────────────── */
        function pickColor(cfg, props) {{
            var v = props[cfg.col];
            if (v === null || v === undefined) return '#aaaaaa';
            if (cfg.type === 'binary')       return BINARY[parseInt(v)]   || '#aaaaaa';
            if (cfg.type === 'movement')     return MOVEMENT[String(v)]   || '#aaaaaa';
            if (cfg.type === 'ground_truth') return GTCOL[String(v)]      || '#aaaaaa';
            return '#aaaaaa';
        }}

        function humanLabel(type, val) {{
            if (val === null || val === undefined) return 'N/A';
            if (type === 'binary')
                return parseInt(val) === 1 ? 'Gentrified' : parseInt(val) === 0 ? 'Not Gentrified' : 'N/A';
            if (type === 'ground_truth') {{
                var GT_LABELS = {{'Low':'Class 0 (likely gentrified)','Medium':'Class 1 (no trend)','High':'Class 2 (unlikely gentrified)'}};
                return GT_LABELS[String(val)] || String(val);
            }}
            return String(val);
        }}

        function buildNbhdLayer(cfg) {{
            var op = OPACITY[cfg.id] !== undefined ? OPACITY[cfg.id] : 0.70;
            return L.geoJson(NBHD_DATA, {{
                style: function (f) {{
                    return {{
                        fillColor  : pickColor(cfg, f.properties),
                        color      : '#555555',
                        weight     : 0.2,
                        fillOpacity: op,
                    }};
                }},
                onEachFeature: (function (c) {{
                    return function (f, lyr) {{
                        var p = f.properties;
                        var avgMean = (p.avg_mean !== null && p.avg_mean !== undefined)
                            ? parseFloat(p.avg_mean).toFixed(2) : 'N/A';
                        var tip = '<b>' + (p.cluster_id || '') + '</b><br>' +
                            c.label + ': <b>' + humanLabel(c.type, p[c.col]) + '</b>';
                        if (c.id === 'gt') tip += '<br>Avg. Score: ' + avgMean;
                        lyr.bindTooltip(tip, {{ sticky: false, direction: 'auto' }});
                    }};
                }})(cfg),
            }});
        }}

        function buildQuartersLayer() {{
            var op = OPACITY['quarters'];
            return L.geoJson(QRTR_DATA, {{
                style: function () {{
                    return {{ fillColor: 'transparent', color: '#333333',
                              weight: 1.25, fillOpacity: 0, opacity: op }};
                }},
                onEachFeature: function (f, lyr) {{
                    lyr.bindTooltip('<b>' + (f.properties.navn || '') + '</b>',
                        {{ sticky: false, direction: 'auto' }});
                }},
            }});
        }}

        /* ── Dynamic legend ─────────────────────────────────────── */
        function updateLegend() {{
            var body = document.getElementById('leg-body');
            if (!body) return;
            var hasBinary   = CONFIGS.some(function(c) {{ return c.type === 'binary'       && !!ACTIVE[c.id]; }});
            var hasMovement = !!ACTIVE['movement'];
            var hasGT       = !!ACTIVE['gt'];
            var hasQuarters = !!ACTIVE['quarters'];
            var sw = function(bg, extra) {{
                return '<span class="sw" style="background:' + bg + (extra||'') + '"></span>';
            }};
            var html = '';
            if (!hasBinary && !hasMovement && !hasGT && !hasQuarters) {{
                html = '<i style="color:#888;font-size:11px">No layers selected</i>';
            }} else {{
                if (hasBinary) {{
                    html += '<div class="lgh">Model Predictions &amp; Soft Ensemble</div>';
                    html += sw('#E15F1D') + 'Gentrified<br>';
                    html += sw('#0C8A00') + 'Not gentrified<br>';
                    html += sw('#aaaaaa') + 'No data<br>';
                }}
                if (hasMovement) {{
                    html += '<div class="lgh">Gentrification Timeline</div>';
                    html += sw('#E15F1D') + 'Already 2020<br>';
                    html += sw('#ED7B40') + 'First: 2025<br>';
                    html += sw('#FBA071') + 'First: 2030<br>';
                    html += sw('#FFCFB6',';border-color:#ccc') + 'First: 2035<br>';
                    html += sw('#0C8A00') + 'Never<br>';
                }}
                if (hasGT) {{
                    html += '<div class="lgh">Ground Truth</div>';
                    html += sw('#E15F1D') + 'Class 2 (unlikely gentrified)<br>';
                    html += sw('#F0E442',';border-color:#ccc') + 'Class 1 (no trend)<br>';
                    html += sw('#0C8A00') + 'Class 0 (likely gentrified)<br>';
                }}
                if (hasQuarters) {{
                    html += '<div class="lgh">Boroughs</div>';
                    html += sw('transparent',';border-color:#333333') + 'Quarter boundary<br>';
                }}
            }}
            body.innerHTML = html;
        }}

        /* ── setOpacity: update live layer + persist for rebuilds ─ */
        window.setOpacity = function (id, val) {{
            OPACITY[id] = val;
            if (!ACTIVE[id]) return;
            if (id === 'quarters') {{
                ACTIVE[id].setStyle({{ opacity: val }});
            }} else {{
                ACTIVE[id].setStyle({{ fillOpacity: val }});
            }}
        }};

        /* ── toggleLayer: called by checkboxes ──────────────────── */
        window.toggleLayer = function (id, checked) {{
            var opRow = document.getElementById('op-row-' + id);
            if (!checked) {{
                if (ACTIVE[id]) {{ leafletMap.removeLayer(ACTIVE[id]); delete ACTIVE[id]; }}
                if (opRow) opRow.style.display = 'none';
                updateLegend();
                return;
            }}
            if (ACTIVE[id]) return;
            var lyr = (id === 'quarters') ? buildQuartersLayer()
                                          : (CFG[id] ? buildNbhdLayer(CFG[id]) : null);
            if (lyr) {{ lyr.addTo(leafletMap); ACTIVE[id] = lyr; }}
            if (opRow) opRow.style.display = 'flex';
            updateLegend();
        }};

        /* ── Defaults ───────────────────────────────────────────── */
        document.getElementById('cb_movement').checked = true;
        document.getElementById('cb_quarters').checked = true;
        window.toggleLayer('quarters', true);   /* quarters first → bottom layer */
        window.toggleLayer('movement', true);   /* movement layer on top */
    }}

    setTimeout(init, 200);
}})();
</script>
"""

# ── Inject HTML + JS ───────────────────────────────────────────────────────────
m.get_root().html.add_child(folium.Element(panel_css + panel_html + legend_html))
m.get_root().html.add_child(folium.Element(js_code))

# ── Save ───────────────────────────────────────────────────────────────────────
out_path = os.path.join(OUT_DIR, 'Web_Map.html')
m.save(out_path)
print(f'✓ Map saved → {os.path.abspath(out_path)}')

Neighbourhood GeoJSON features : 2232
Quarter GeoJSON features       : 18
✓ Map saved → c:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\Web_Map.html
